In [1]:
import json
import re
from typing import List, Dict, Any

# Load the JSON data
def load_processed_data(filepath: str) -> Dict[str, Any]:
    with open(filepath, 'r') as f:
        data = json.load(f)
    return data

# --- Configuration ---
PROCESSED_JSON_PATH = "outputs/NIGERIA_TAX_ACT_2025_processed.json"
# ---

data = load_processed_data(PROCESSED_JSON_PATH)
print(f"Loaded document: {data['document_metadata']['title']}")
print(f"Total sections: {data['summary']['total_sections']}")
print(f"Total tables: {data['summary']['total_tables']}")
print(f"Total definitions: {data['summary']['total_definitions']}")

Loaded document: Nigeria Tax Act 2025
Total sections: 0
Total tables: 33
Total definitions: 227


In [2]:
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

@dataclass
class Chunk:
    id: str
    type: str
    content: str
    chapter: Optional[str] = None
    part: Optional[str] = None
    section_number: Optional[str] = None
    section_title: Optional[str] = None
    subsection_reference: Optional[str] = None
    parent_section_id: Optional[str] = None
    page_number: Optional[int] = None
    tokens: Optional[int] = None
    related_table_ids: List[str] = field(default_factory=list)
    related_definition_terms: List[str] = field(default_factory=list)

    def to_dict(self):
        return asdict(self)

# Add these so your rehydration calls work:
@dataclass
class Section:
    id: str
    number: Optional[str]
    title: Optional[str]
    content: str
    page_number: Optional[int] = None

@dataclass
class Table:
    id: str
    title: Optional[str]
    headers: List[str]
    rows: List[List[str]]
    page_number: Optional[int] = None
    section_ref: Optional[str] = None
    location: Optional[str] = None  # accept incoming 'location' key
    metadata: Optional[Dict[str, Any]] = None  # handle unexpected 'metadata' key

    def __post_init__(self):
        # If the source used 'location' for a page number, try to coerce it
        if self.page_number is None and self.location is not None:
            try:
                self.page_number = int(self.location)
            except Exception:
                pass

In [3]:
import re

def parse_subsections(section_content: str):
    """
    Splits section content based on patterns like (1), (2), (a), (b).
    Returns a list of dictionaries, each with a 'reference' and 'text'.
    """
    # This regex looks for patterns like (1), (2), (a), (b), (i), (ii) at the start of a line or after a newline.
    # It's a complex task, so this is a simplified but effective version.
    subsection_pattern = re.compile(r'\n\s*\(([a-z0-9]+)\)\s+', re.IGNORECASE)
    parts = subsection_pattern.split(section_content)

    subsections = []
    # The first part is always the text before the first subsection marker
    if len(parts) > 1:
        # The structure is: [intro, marker1, text1, marker2, text2, ...]
        subsections.append({"reference": "intro", "text": parts[0].strip()})
        for i in range(1, len(parts), 2):
            if i+1 < len(parts):
                subsections.append({"reference": f"({parts[i]})", "text": parts[i+1].strip()})
    else:
        # No subsections found, treat the whole thing as one chunk.
        subsections.append({"reference": "full", "text": section_content.strip()})

    # Filter out empty subsections
    return [s for s in subsections if s['text']]


In [4]:
def create_chunks_from_data(data: Dict[str, Any]) -> List[Chunk]:
    chunks = []
    chunk_counter = 0

    # Create a quick lookup for tables by page number or section reference
    # This is a simplification. A more robust method would try to match table titles or surrounding text.
    tables_by_page = {}
    for table in data['tables']:
        page = table.get('page_number')
        if page:
            if page not in tables_by_page:
                tables_by_page[page] = []
            tables_by_page[page].append(table)

    # Process each section
    for section_dict in data['sections']:
        section = Section(**section_dict) # Rehydrate the dataclass

        # --- 1. Create a chunk for the whole section (if it's short) ---
        # Or we can always split into subsections. Let's split into subsections.
        subsections = parse_subsections(section.content)

        # Find related tables on the same page
        related_tables = tables_by_page.get(section.page_number, [])

        for sub in subsections:
            chunk_counter += 1
            chunk_id = f"chunk_{chunk_counter:05d}"

            # Construct the content, including the section title for context
            if sub['reference'] == "intro":
                content = f"{section.title}\n\n{sub['text']}"
                subsection_ref_str = "Introduction"
            elif sub['reference'] == "full":
                content = f"{section.title}\n\n{sub['text']}"
                subsection_ref_str = "Full Section"
            else:
                content = f"{section.title} {sub['reference']}\n\n{sub['text']}"
                subsection_ref_str = sub['reference']

            # Estimate tokens (roughly 4 chars per token)
            estimated_tokens = len(content) // 4

            # Create the chunk
            chunk = Chunk(
                id=chunk_id,
                type="subsection" if sub['reference'] not in ["intro", "full"] else "section",
                content=content,
                chapter=None,  # We need to infer this from the structure
                part=None,     # We need to infer this from the structure
                section_number=section.number,
                section_title=section.title,
                subsection_reference=subsection_ref_str,
                parent_section_id=section.id,
                page_number=section.page_number,
                tokens=estimated_tokens,
                related_table_ids=[t['id'] for t in related_tables if t.get('section_ref') == section.id or t.get('page_number') == section.page_number]
            )
            chunks.append(chunk)

    # --- 2. Create chunks for definitions (if you want to index them separately) ---
    for term, definition in data['definitions'].items():
        chunk_counter += 1
        chunk_id = f"chunk_{chunk_counter:05d}"
        content = f"Definition of {term}: {definition}"
        chunk = Chunk(
            id=chunk_id,
            type="definition",
            content=content,
            subsection_reference=f"Definition of {term}",
            tokens=len(content) // 4
        )
        chunks.append(chunk)

    # --- 3. Create chunks for tables (if you want to index them separately) ---
    # We might also create chunks that combine the table with its preceding section text.
    for table_dict in data['tables']:
        table = Table(**table_dict)
        chunk_counter += 1
        chunk_id = f"chunk_{chunk_counter:05d}"
        # Convert table to a readable string format
        table_str = f"Table: {table.title}\n"
        table_str += " | ".join(table.headers) + "\n"
        for row in table.rows[:10]: # Limit to first 10 rows to keep chunk size manageable
            table_str += " | ".join(row) + "\n"
        if len(table.rows) > 10:
            table_str += f"... and {len(table.rows) - 10} more rows"

        chunk = Chunk(
            id=chunk_id,
            type="table",
            content=table_str,
            subsection_reference=f"Table on page {table.page_number}",
            page_number=table.page_number,
            tokens=len(table_str) // 4
        )
        chunks.append(chunk)


    print(f"Created {len(chunks)} chunks.")
    return chunks

# Run the chunking
all_chunks = create_chunks_from_data(data)

# Let's look at a sample chunk
if all_chunks:
    print("\n--- Sample Chunk ---")
    sample = all_chunks[0]
    print(f"ID: {sample.id}")
    print(f"Type: {sample.type}")
    print(f"Section: {sample.section_number} - {sample.section_title}")
    print(f"Subsection: {sample.subsection_reference}")
    print(f"Content Preview: {sample.content[:300]}...")

Created 260 chunks.

--- Sample Chunk ---
ID: chunk_00001
Type: definition
Section: None - None
Subsection: Definition of money instruments
Content Preview: Definition of money instruments: instruments traded in money markets including government securities, treasury Acts, treasury or savings certificates, debenture certificates, commercial papers, certificates of deposits, call money, commercial Acts, treasury bonds and any other money instrument...


In [5]:
from pathlib import Path

# Convert chunks to dictionaries
chunks_dict = [chunk.to_dict() for chunk in all_chunks]

# Save to a new JSON file
output_chunk_path = Path(PROCESSED_JSON_PATH).parent / f"{Path(PROCESSED_JSON_PATH).stem}_chunks.json"
with open(output_chunk_path, 'w') as f:
    json.dump(chunks_dict, f, indent=2)

print(f"\n💾 Saved chunks to: {output_chunk_path}")
print(f"📊 Total chunks: {len(chunks_dict)}")


💾 Saved chunks to: outputs\NIGERIA_TAX_ACT_2025_processed_chunks.json
📊 Total chunks: 260


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks_dict).tolist()

c:\Users\SITOG-External Sales\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\SITOG-External Sales\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SITOG-External Sales\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you ei

TypeError: 'Chunk' object is not subscriptable